## Installing the correct version of transformers

In [ ]:
!pip uninstall -y trl transformers peft accelerate
!pip install -q transformers==4.47.1 peft==0.13.2 accelerate==1.2.1 datasets sqlparse sentencepiece

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.6 MB/s eta 0:00:00


In [ ]:
import transformers
print(transformers.__version__)

4.47.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Mount Google Drive and Create Persistent Folders

For Milestone 1, all important outputs are saved directly to Google Drive rather than the temporary Colab runtime.

This ensures that:
- training checkpoints are preserved across disconnects,
- CSV outputs and adapters remain available later,
- and the experiment can be resumed or reused in future notebooks.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os

drive.mount('/content/drive')

DRIVE_ROOT = Path("/content/drive/MyDrive/CS_512_Project")
MILESTONE1_DIR = DRIVE_ROOT / "milestone1_lora"
CHECKPOINT_DIR = MILESTONE1_DIR / "checkpoints"
ADAPTER_DIR = MILESTONE1_DIR / "final_adapter"
CSV_DIR = MILESTONE1_DIR / "csv_outputs"
CONFIG_DIR = MILESTONE1_DIR / "run_configs"

for p in [DRIVE_ROOT, MILESTONE1_DIR, CHECKPOINT_DIR, ADAPTER_DIR, CSV_DIR, CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Milestone 1 directory:", MILESTONE1_DIR)

Mounted at /content/drive
Drive root: /content/drive/MyDrive/CS_512_Project
Milestone 1 directory: /content/drive/MyDrive/CS_512_Project/milestone1_lora


## 2. Verify GPU Availability

Before training, we confirm that the notebook is running on an A100 and that PyTorch can use CUDA correctly.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Device count:", torch.cuda.device_count())

CUDA available: True
GPU name: NVIDIA A100-SXM4-80GB
Device count: 1


## 3. Save the Run Configuration

We store the training setup as a JSON file so that the run remains reproducible and easy to reference later.

In [ ]:
# #Optimized training config for less powerful GPU

# run_config = {
#     "model_name": "microsoft/Phi-3.5-mini-instruct",
#     "train_limit": 200,
#     "dev_limit": 50,
#     "max_len": 768,
#     "lora_r": 8,
#     "lora_alpha": 16,
#     "lora_dropout": 0.05,
#     "learning_rate": 2e-4,
#     "num_train_epochs": 1,
#     "per_device_train_batch_size": 1,
#     "gradient_accumulation_steps": 4,
#     "target_modules": ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"]
# }

# config_path = CONFIG_DIR / "milestone1_run_config.json"
# with open(config_path, "w") as f:
#     json.dump(run_config, f, indent=2)

# print("Saved run config to:", config_path)

In [ ]:
#Training config for powerful GPUs like A100

run_config = {
    "model_name": "microsoft/Phi-3.5-mini-instruct",
    "train_limit": 2500,
    "dev_limit": 200,
    "max_len": 768,
    "lora_r": 32,
    "lora_alpha": 64,          # Keep alpha = 2*r for standard scaling
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,     # CHANGED: 1e-4 → 2e-4 (standard LoRA SFT rate)
    "num_train_epochs": 4,     # CHANGED: 2 → 3 epochs (more passes with correct format)
    "per_device_train_batch_size": 2,    # CHANGED: 1 → 2 (if A100 VRAM allows)
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,   # CHANGED: 8 → 4 (effective batch = 8, same)
    "target_modules": ["qkv_proj", "o_proj", "gate_up_proj", "down_proj", "embed_tokens", "lm_head"],
    "precision": "bf16",
    "warmupratio": 0.05,
    "weightdecay": 0.01,
    "loggingsteps": 10,
    "evalsteps": 100,
    "savesteps": 100,
}

config_path = CONFIG_DIR / "milestone1_run_config.json"
with open(config_path, "w") as f:
    json.dump(run_config, f, indent=2)

print("Saved config to:", config_path)

Saved config to: /content/drive/MyDrive/CS_512_Project/milestone1_lora/run_configs/milestone1_run_config.json


## 4. Load Preprocessed Spider Data from Google Drive

This notebook expects preprocessed Spider examples to already exist as saved files in Google Drive.

We load those saved artifacts here so that Milestone 1 can start directly from train-ready examples.

In [ ]:
import json
from pathlib import Path

#Check if the files exist on the drive

PROCESSED_DIR = DRIVE_ROOT / "processed_data"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_train_path = PROCESSED_DIR / "processed_train.json"
processed_dev_path = PROCESSED_DIR / "processed_dev.json"

print("processed_train path:", processed_train_path)
print("processed_dev path:", processed_dev_path)
print("train exists:", processed_train_path.exists())
print("dev exists:", processed_dev_path.exists())

#Load the files and store them into python variables

with open(processed_train_path, "r", encoding="utf-8") as f:
    processed_train = json.load(f)

with open(processed_dev_path, "r", encoding="utf-8") as f:
    processed_dev = json.load(f)

print("Loaded processed_train:", len(processed_train))#############
print("Loaded processed_dev:", len(processed_dev))#############
print("Sample keys:", processed_train[0].keys())


processed_train path: /content/drive/MyDrive/CS_512_Project/processed_data/processed_train.json
processed_dev path: /content/drive/MyDrive/CS_512_Project/processed_data/processed_dev.json
train exists: True
dev exists: True
Loaded processed_train: 2500
Loaded processed_dev: 1034
Sample keys: dict_keys(['db_id', 'question', 'prompt', 'target_sql', 'schema_text', 'sqlite_path', 'text'])


In [ ]:
# ── Fix stale sqlite_path values baked into the preprocessed JSON ──────────
# The Baseline notebook saved paths rooted at /content/text2sql_project/
# which no longer exists in this session. Remap them to the current
# Spider data location on Drive.

SPIDER_LOCAL_ROOT = DRIVE_ROOT / "spider" / "spider_data"   # adjust if your Drive path differs

OLD_PREFIX = "/content/text2sql_project/spider_data"
NEW_PREFIX = "/content/drive/MyDrive/CS_512_Project/spider-20260426T221853Z-3-001/spider/spider_data"

def fix_sqlite_path(example):
    old_path = example["sqlite_path"]
    if old_path.startswith(OLD_PREFIX):
        example["sqlite_path"] = old_path.replace(OLD_PREFIX, NEW_PREFIX, 1)
    return example

processed_train = [fix_sqlite_path(ex) for ex in processed_train]
processed_dev   = [fix_sqlite_path(ex) for ex in processed_dev]

# Verify
import os
sample_path = processed_dev[0]["sqlite_path"]
print("Remapped path:", sample_path)
print("File exists:", os.path.exists(sample_path))

Remapped path: /content/drive/MyDrive/CS_512_Project/spider-20260426T221853Z-3-001/spider/spider_data/database/concert_singer/concert_singer.sqlite
File exists: True


## 5. Prepare the Spider Training and Dev Subsets

For Milestone 1, we use a larger subset than the smoke-test version so the LoRA fine-tuning step becomes more meaningful.

In [ ]:
TRAIN_LIMIT = run_config["train_limit"]
DEV_LIMIT = run_config["dev_limit"]

train_subset = processed_train[:TRAIN_LIMIT]
dev_subset = processed_dev[:DEV_LIMIT]

print("Train subset size:", len(train_subset))
print("Dev subset size:", len(dev_subset))
print("Example question:", train_subset[0]["question"])
print("Example gold SQL:", train_subset[0]["target_sql"])

Train subset size: 2500
Dev subset size: 200
Example question: How many heads of the departments are older than 56 ?
Example gold SQL: SELECT count(*) FROM head WHERE age  >  56


In [ ]:
print(processed_train[0].keys())

dict_keys(['db_id', 'question', 'prompt', 'target_sql', 'schema_text', 'sqlite_path', 'text'])




## 6. Load the Tokenizer

We load the tokenizer that matches the base Phi-3.5 Mini Instruct model.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = run_config["model_name"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded for:", MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Tokenizer loaded for: microsoft/Phi-3.5-mini-instruct


## 7. Convert Examples into Supervised Fine-Tuning Format

Each training example is formatted using the model's own chat template (system → user → assistant), matching the format the model was originally trained on. This is critical for instruction-tuned models like Phi-3.5-mini-instruct — training on raw concatenated text causes a train/inference format mismatch that prevents learning.

In [ ]:

from datasets import Dataset

def make_sft_record(example):
    messages = [
        {
            "role": "system",
            "content": "You are a text-to-SQL assistant. Return only valid SQLite SQL."
        },
        {
            "role": "user",
            "content": example["prompt"]
        },
        {
            "role": "assistant",
            "content": example["target_sql"]
        },
    ]
    # Apply the same chat template used at inference time
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,   # False because we include the answer
    )
    return {"text": text}

train_sft = [make_sft_record(x) for x in train_subset]
dev_sft   = [make_sft_record(x) for x in dev_subset]

train_ds = Dataset.from_list(train_sft)
dev_ds   = Dataset.from_list(dev_sft)

print(train_ds[0]["text"][:1200])

<|system|>
You are a text-to-SQL assistant. Return only valid SQLite SQL.<|end|>
<|user|>
Generate a valid SQLite SQL query for the given question using only the provided schema.
Return only SQL.

Schema:
Tables:
- department(Department_ID, Name, Creation, Ranking, Budget_in_Billions, Num_Employees)
- head(head_ID, name, born_state, age)
- management(department_ID, head_ID, temporary_acting)

Primary Keys:
- department.Department_ID
- head.head_ID
- management.department_ID

Foreign Keys:
- management.head_ID -> head.head_ID
- management.department_ID -> department.Department_ID

Question:
How many heads of the departments are older than 56 ?

SQL:<|end|>
<|assistant|>
SELECT count(*) FROM head WHERE age  >  56<|end|>
<|endoftext|>


## 8. Tokenize the Training Data

We tokenize the training and dev sets without padding — dynamic padding is handled by the custom data collator at batch time. This reduces wasted computation on short sequences. Truncation is applied at MAX_LEN to handle long schemas.

In [ ]:
MAX_LEN = run_config["max_len"]

def tokenize_function(batch):
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,          # Do NOT pad here — collator handles it per-batch
    )
    return encoded

tokenized_train = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dev   = dev_ds.map(tokenize_function,   batched=True, remove_columns=["text"])

# Set format so Trainer gets tensors
# tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask"])
# tokenized_dev.set_format(type="torch",   columns=["input_ids", "attention_mask"])

print(tokenized_train)
print("Sample input_ids length:", len(tokenized_train[0]["input_ids"]))

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2500
})
Sample input_ids length: 219


## 9. Load a Fresh Base Model for LoRA Training

We reload the base model specifically for fine-tuning and configure it to use BF16.

In [ ]:
from transformers import AutoModelForCausalLM

train_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

train_model.config.use_cache = False

print("Loaded training model:", MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Loaded training model: microsoft/Phi-3.5-mini-instruct


## 10. Attach LoRA Adapters

We add trainable LoRA adapters to selected projection layers while keeping the original base-model weights frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=run_config["lora_r"],
    lora_alpha=run_config["lora_alpha"],
    lora_dropout=run_config["lora_dropout"],
    bias="none",
    target_modules=run_config["target_modules"],
)

train_model = get_peft_model(train_model, lora_config)
train_model.print_trainable_parameters()

trainable params: 52,580,352 || all params: 3,873,659,904 || trainable%: 1.3574


## 11. Enable Gradient Checkpointing

Gradient checkpointing can still help reduce memory use for longer sequences and larger effective batches.

In [ ]:
train_model.config.use_cache = False

if hasattr(train_model, "enable_input_require_grads"):
    train_model.enable_input_require_grads()

if hasattr(train_model, "gradient_checkpointing_enable"):
    train_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

print("Gradient checkpointing enabled.")
print("use_cache:", train_model.config.use_cache)

Gradient checkpointing enabled.
use_cache: False


## 12. Configure Trainer and Save Checkpoints to Google Drive

The training checkpoints are written directly to Google Drive so they remain available after the notebook session ends.

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=run_config["per_device_train_batch_size"],
    per_device_eval_batch_size=run_config["per_device_eval_batch_size"],
    gradient_accumulation_steps=run_config["gradient_accumulation_steps"],
    num_train_epochs=run_config["num_train_epochs"],
    learning_rate=run_config["learning_rate"],
    weight_decay=run_config["weightdecay"],
    warmup_ratio=run_config["warmupratio"],
    logging_steps=run_config["loggingsteps"],
    eval_strategy="steps",
    eval_steps=run_config["evalsteps"],
    save_strategy="steps",
    save_steps=run_config["savesteps"],
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
    disable_tqdm=True,
)

In [ ]:
trainable = [(n, p.numel()) for n, p in train_model.named_parameters() if p.requires_grad]
print("Number of trainable parameter tensors:", len(trainable))
print("First 10 trainable tensors:")
for name, count in trainable[:10]:
    print(name, count)

Number of trainable parameter tensors: 260
First 10 trainable tensors:
base_model.model.model.embed_tokens.lora_embedding_A.default 1026048
base_model.model.model.embed_tokens.lora_embedding_B.default 98304
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight 98304
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight 98304
base_model.model.model.layers.0.self_attn.qkv_proj.lora_A.default.weight 98304
base_model.model.model.layers.0.self_attn.qkv_proj.lora_B.default.weight 294912
base_model.model.model.layers.0.mlp.gate_up_proj.lora_A.default.weight 98304
base_model.model.model.layers.0.mlp.gate_up_proj.lora_B.default.weight 524288
base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight 262144
base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight 98304


## 13. Run Milestone 1 LoRA Fine-Tuning

This is the actual fine-tuning step for Milestone 1.

**Adding Plain-Text Training Logs**

This setup prints readable training updates directly in the notebook so progress is visible even when the Colab progress bar does not render.

In [ ]:
from transformers import TrainerCallback, TrainingArguments, Trainer, DataCollatorForLanguageModeling

class StepPrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        step = state.global_step
        epoch = state.epoch

        pieces = [f"step={step}"]
        if epoch is not None:
            pieces.append(f"epoch={epoch:.3f}")

        for key in ["loss", "eval_loss", "learning_rate"]:
            if key in logs:
                value = logs[key]
                if isinstance(value, float):
                    pieces.append(f"{key}={value:.6f}")
                else:
                    pieces.append(f"{key}={value}")

        print(" | ".join(pieces))

**Attach the Callback to the Trainer**

We pass the callback and a custom data collator into the Trainer. The custom collator is critical: it masks all prompt tokens in the labels so the cross-entropy loss is computed only on the SQL output tokens. Without this, the model wastes gradient capacity trying to memorize prompt formatting instead of learning SQL generation.

In [ ]:
from transformers import Trainer, DataCollatorForSeq2Seq
import torch

MAX_LEN = run_config["max_len"]

# Re-tokenize with labels properly masked
# For each example: input_ids = full sequence, labels = -100 for prompt, SQL tokens for answer

def tokenize_with_labels(example):
    """
    Tokenize the full chat-formatted text, then mask the prompt portion in labels.
    We find the assistant turn by tokenizing just the prompt portion and using its length.
    """
    full_text = example["text"]

    # Tokenize full sequence (prompt + SQL answer)
    full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )

    # Find where the assistant answer starts by tokenizing everything
    # UP TO the <|assistant|> marker, then measuring that length
    # Split on the assistant marker to isolate the prompt portion
    split_marker = "<|assistant|>\n"
    if split_marker in full_text:
        prompt_part = full_text[:full_text.rfind(split_marker) + len(split_marker)]
    else:
        prompt_part = full_text  # fallback: mask everything (won't learn but won't crash)

    prompt_tokens = tokenizer(
        prompt_part,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    prompt_len = len(prompt_tokens["input_ids"])

    # Labels: -100 for prompt tokens, real token IDs for SQL answer tokens
    labels = [-100] * prompt_len + full["input_ids"][prompt_len:]
    # Clip to same length as input_ids (in case of truncation edge case)
    labels = labels[:len(full["input_ids"])]

    full["labels"] = labels
    return full

# Re-run tokenization with labels
tokenized_train = train_ds.map(tokenize_with_labels, batched=False, remove_columns=["text"])
tokenized_dev   = dev_ds.map(tokenize_with_labels,   batched=False, remove_columns=["text"])

print("Sample label check (first example):")
sample_labels = tokenized_train[0]["labels"]
non_masked = [l for l in sample_labels if l != -100]
print(f"  Total tokens: {len(sample_labels)}")
print(f"  Masked (-100): {sample_labels.count(-100)}")
print(f"  Active (SQL) tokens: {len(non_masked)}")
print(f"  First 10 active token IDs: {non_masked[:10]}")
print(f"  Decoded SQL portion: {tokenizer.decode(non_masked[:50])}")

# Use DataCollatorForSeq2Seq — handles dynamic padding and respects -100 labels
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=train_model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=train_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator,
    callbacks=[StepPrinterCallback()],
)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Sample label check (first example):
  Total tokens: 219
  Masked (-100): 205
  Active (SQL) tokens: 14
  First 10 active token IDs: [5097, 2302, 22798, 3895, 2343, 5754, 5046, 29871, 1405, 259]
  Decoded SQL portion: SELECT count(*) FROM head WHERE age  >  56<|end|><|endoftext|>


***Start fine-tuning training***

In [ ]:
trainer.train()

step=10 | epoch=0.032 | loss=11.982500 | learning_rate=0.000032
{'loss': 11.9825, 'grad_norm': 167.18032836914062, 'learning_rate': 3.1746031746031745e-05, 'epoch': 0.032}
step=20 | epoch=0.064 | loss=5.290600 | learning_rate=0.000063
{'loss': 5.2906, 'grad_norm': 14.23044490814209, 'learning_rate': 6.349206349206349e-05, 'epoch': 0.064}
step=30 | epoch=0.096 | loss=1.794900 | learning_rate=0.000095
{'loss': 1.7949, 'grad_norm': 9.357778549194336, 'learning_rate': 9.523809523809524e-05, 'epoch': 0.096}
step=40 | epoch=0.128 | loss=0.952600 | learning_rate=0.000127
{'loss': 0.9526, 'grad_norm': 22.01067352294922, 'learning_rate': 0.00012698412698412698, 'epoch': 0.128}
step=50 | epoch=0.160 | loss=0.870800 | learning_rate=0.000159
{'loss': 0.8708, 'grad_norm': 16.947628021240234, 'learning_rate': 0.00015873015873015873, 'epoch': 0.16}
step=60 | epoch=0.192 | loss=0.782000 | learning_rate=0.000190
{'loss': 0.782, 'grad_norm': 10.48423957824707, 'learning_rate': 0.00019047619047619048, 'e

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:227: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


step=110 | epoch=0.352 | loss=0.535200 | learning_rate=0.000192
{'loss': 0.5352, 'grad_norm': 11.556078910827637, 'learning_rate': 0.00019206751054852322, 'epoch': 0.352}
step=120 | epoch=0.384 | loss=0.565100 | learning_rate=0.000190
{'loss': 0.5651, 'grad_norm': 26.01801872253418, 'learning_rate': 0.00019037974683544303, 'epoch': 0.384}
step=130 | epoch=0.416 | loss=0.503200 | learning_rate=0.000189
{'loss': 0.5032, 'grad_norm': 7.931046485900879, 'learning_rate': 0.00018869198312236287, 'epoch': 0.416}
step=140 | epoch=0.448 | loss=0.583200 | learning_rate=0.000187
{'loss': 0.5832, 'grad_norm': 5.05265474319458, 'learning_rate': 0.0001870042194092827, 'epoch': 0.448}
step=150 | epoch=0.480 | loss=0.594500 | learning_rate=0.000185
{'loss': 0.5945, 'grad_norm': 11.42601203918457, 'learning_rate': 0.00018531645569620255, 'epoch': 0.48}
step=160 | epoch=0.512 | loss=0.515900 | learning_rate=0.000184
{'loss': 0.5159, 'grad_norm': 9.286465644836426, 'learning_rate': 0.00018362869198312238

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:227: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


step=210 | epoch=0.672 | loss=0.353800 | learning_rate=0.000175
{'loss': 0.3538, 'grad_norm': 13.661434173583984, 'learning_rate': 0.00017518987341772152, 'epoch': 0.672}
step=220 | epoch=0.704 | loss=0.465300 | learning_rate=0.000174
{'loss': 0.4653, 'grad_norm': 11.035484313964844, 'learning_rate': 0.00017350210970464136, 'epoch': 0.704}
step=230 | epoch=0.736 | loss=0.463800 | learning_rate=0.000172
{'loss': 0.4638, 'grad_norm': 12.522775650024414, 'learning_rate': 0.0001718143459915612, 'epoch': 0.736}
step=240 | epoch=0.768 | loss=0.426300 | learning_rate=0.000170
{'loss': 0.4263, 'grad_norm': 5.441831111907959, 'learning_rate': 0.000170126582278481, 'epoch': 0.768}
step=250 | epoch=0.800 | loss=0.405400 | learning_rate=0.000168
{'loss': 0.4054, 'grad_norm': 7.78806209564209, 'learning_rate': 0.00016843881856540085, 'epoch': 0.8}
step=260 | epoch=0.832 | loss=0.421700 | learning_rate=0.000167
{'loss': 0.4217, 'grad_norm': 3.636899471282959, 'learning_rate': 0.0001667510548523207, 

...

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:227: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


step=910 | epoch=2.909 | loss=0.085000 | learning_rate=0.000057
{'loss': 0.085, 'grad_norm': 7.6767897605896, 'learning_rate': 5.7046413502109705e-05, 'epoch': 2.9088000000000003}
step=920 | epoch=2.941 | loss=0.110400 | learning_rate=0.000055
{'loss': 0.1104, 'grad_norm': 43.70481491088867, 'learning_rate': 5.535864978902954e-05, 'epoch': 2.9408}
step=930 | epoch=2.973 | loss=0.084900 | learning_rate=0.000054
{'loss': 0.0849, 'grad_norm': 5.1291117668151855, 'learning_rate': 5.367088607594937e-05, 'epoch': 2.9728}
step=940 | epoch=3.003 | loss=0.069600 | learning_rate=0.000052
{'loss': 0.0696, 'grad_norm': 0.9250327348709106, 'learning_rate': 5.19831223628692e-05, 'epoch': 3.0032}
step=950 | epoch=3.035 | loss=0.064300 | learning_rate=0.000050
{'loss': 0.0643, 'grad_norm': 1.2110283374786377, 'learning_rate': 5.029535864978904e-05, 'epoch': 3.0352}
step=960 | epoch=3.067 | loss=0.056000 | learning_rate=0.000049
{'loss': 0.056, 'grad_norm': 40.33066177368164, 'learning_rate': 4.8607594

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:227: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


step=1010 | epoch=3.227 | loss=0.054400 | learning_rate=0.000040
{'loss': 0.0544, 'grad_norm': 31.692546844482422, 'learning_rate': 4.016877637130802e-05, 'epoch': 3.2272}
step=1020 | epoch=3.259 | loss=0.088300 | learning_rate=0.000038
{'loss': 0.0883, 'grad_norm': 0.2614010274410248, 'learning_rate': 3.848101265822785e-05, 'epoch': 3.2592}
step=1030 | epoch=3.291 | loss=0.053400 | learning_rate=0.000037
{'loss': 0.0534, 'grad_norm': 6.704862594604492, 'learning_rate': 3.679324894514768e-05, 'epoch': 3.2912}
step=1040 | epoch=3.323 | loss=0.032300 | learning_rate=0.000035
{'loss': 0.0323, 'grad_norm': 0.7412715554237366, 'learning_rate': 3.5105485232067516e-05, 'epoch': 3.3232}
step=1050 | epoch=3.355 | loss=0.022300 | learning_rate=0.000033
{'loss': 0.0223, 'grad_norm': 2.7774040699005127, 'learning_rate': 3.341772151898735e-05, 'epoch': 3.3552}


KeyboardInterrupt: 

## 14. Save the Final LoRA Adapter and Tokenizer

After training, we save the final adapter and tokenizer to Google Drive. Saving to Drive (not local /content/) is mandatory — Colab runtime memory is wiped after training completes or the session restarts. We also define OUTPUTDIR here so downstream CSV-saving cells have a valid path.

In [ ]:
from pathlib import Path

# Save to Google Drive so adapter survives runtime resets
ADAPTER_DRIVE_DIR = MILESTONE1_DIR / "final_adapter"
ADAPTER_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(ADAPTER_DRIVE_DIR))
tokenizer.save_pretrained(str(ADAPTER_DRIVE_DIR))

# Also define OUTPUTDIR for later CSV cells (Cell 56, 58, 60)
OUTPUTDIR = MILESTONE1_DIR / "csv_outputs"
OUTPUTDIR.mkdir(parents=True, exist_ok=True)

# Update ADAPTER_DIR so Cell 41 loads from the same Drive path
ADAPTER_DIR = ADAPTER_DRIVE_DIR

print("Saved adapter to:", ADAPTER_DIR)
print("Saved files:", sorted([p.name for p in ADAPTER_DIR.iterdir()])[:10])

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:227: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Saved adapter to: /content/drive/MyDrive/CS_512_Project/milestone1_lora/final_adapter
Saved files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json', 'training_args.bin']


## 15. Reload the Base Model and Attach the Saved LoRA Adapter

We now load a clean base model and attach the saved LoRA adapter for inference-time evaluation.

In [ ]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

if "trainer" in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

# Load tokenizers
eval_tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

# ── Model 1: Pure base model, NO adapter ─────────────────────────────────────
base_model_for_compare = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="eager",
)
base_model_for_compare.eval()

# ── Model 2: Fresh base model + LoRA adapter ─────────────────────────────────
eval_base_for_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="eager",
)
eval_model = PeftModel.from_pretrained(eval_base_for_lora, str(ADAPTER_DIR))
eval_model.eval()

# Sanity check — must print False
print("Same object?", base_model_for_compare is eval_model)
print("Base model type:", type(base_model_for_compare))
print("Tuned model type:", type(eval_model))
print("Reloaded both models successfully.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Same object? False
Base model type: <class 'transformers_modules.microsoft.Phi-3.5-mini-instruct.2fe192450127e6a83f7441aef6e3ca586c338b77.modeling_phi3.Phi3ForCausalLM'>
Tuned model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Reloaded both models successfully.


## 16. Build a Base-Model Comparison Helper

We define the inference helper that uses the model's chat template, matching the training format. Both the base model and the fine-tuned model are called through the same function, ensuring a fair comparison. The only variable is whether LoRA adapter weights are active.

In [ ]:
import torch
import re

def clean_sql_output(text: str) -> str:
    text = text.strip()
    if text.lower().startswith("sql"):
        text = text[3:].strip(": \n")
    if "```" in text:
        parts = text.split("```")
        text = parts[1] if len(parts) >= 2 else parts[0]
        text = re.sub(r"^sql\s*", "", text.strip(), flags=re.IGNORECASE)
    text = text.split("\n\n")[0].strip()
    text = text.split(";--")[0].strip()
    return text

#base_model_for_compare  = eval_base_model
base_tokenizer_for_compare = tokenizer   # original tokenizer loaded in Cell 19

def generate_sql_with_model(prompt, gen_model, gen_tokenizer, max_new_tokens=128):
    """
    Generate SQL using the model's chat template for proper instruction-following.
    This must match the format used during fine-tuning (after the fix in Cell 17).
    """
    messages = [
        {
            "role": "system",
            "content": "You are a text-to-SQL assistant. Return only valid SQLite SQL."
        },
        {
            "role": "user",
            "content": prompt
        },
    ]

    chat_inputs = gen_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,   # True at inference — we want the model to continue
        return_tensors="pt",
        return_dict=True,
    )

    chat_inputs = {k: v.to(gen_model.device) for k, v in chat_inputs.items()}

    if "attention_mask" not in chat_inputs:
        chat_inputs["attention_mask"] = torch.ones_like(chat_inputs["input_ids"])

    prompt_len = chat_inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = gen_model.generate(
            **chat_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=gen_tokenizer.eos_token_id,
            eos_token_id=gen_tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_ids = outputs[0, prompt_len:]
    decoded = gen_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return clean_sql_output(decoded)

print("base_model type:", type(base_model_for_compare))
print("eval_model type:", type(eval_model))
print("Are they the same object?", base_model_for_compare is eval_model)

base_model type: <class 'transformers_modules.microsoft.Phi-3.5-mini-instruct.2fe192450127e6a83f7441aef6e3ca586c338b77.modeling_phi3.Phi3ForCausalLM'>
eval_model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Are they the same object? False


## 17. Qualitative Check on a Few Dev Examples

We inspect a small sample of dev questions and compare the gold SQL, the original base-model output, and the fine-tuned LoRA output.

In [ ]:
sample_count = 5
validation_examples = processed_dev[:sample_count] if "processed_dev" in globals() else processed_dev[:sample_count]

milestone1_rows = []

for i, ex in enumerate(validation_examples):
    prompt = ex["prompt"]

    base_sql = generate_sql_with_model(
        prompt,
        base_model_for_compare,
        base_tokenizer_for_compare,
    )

    tuned_sql = generate_sql_with_model(
        prompt,
        eval_model,
        eval_tokenizer,
    )

    row = {
        "example_id": i,
        "db_id": ex["db_id"],
        "question": ex["question"],
        "gold_sql": ex["target_sql"],
        "base_sql": base_sql,
        "tuned_sql": tuned_sql,
        "sqlite_path": ex["sqlite_path"],
    }
    milestone1_rows.append(row)

    print("=" * 100)
    print("Example:", i)
    print("DB ID:", ex["db_id"])
    print("Question:", ex["question"])
    print("\nGold SQL:\n", ex["target_sql"])
    print("\nBase model SQL:\n", base_sql)
    print("\nTuned model SQL:\n", tuned_sql)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


Example: 0
DB ID: concert_singer
Question: How many singers do we have?

Gold SQL:
 SELECT count(*) FROM singer

Base model SQL:
 SELECT COUNT(DISTINCT singer.Singer_ID) AS NumberOfSingers
FROM singer;

Tuned model SQL:
 SELECT count(*) FROM singer
Example: 1
DB ID: concert_singer
Question: What is the total number of singers?

Gold SQL:
 SELECT count(*) FROM singer

Base model SQL:
 SELECT COUNT(DISTINCT singer.Singer_ID) AS total_singers
FROM singer;

Tuned model SQL:
 SELECT count(*) FROM singer
Example: 2
DB ID: concert_singer
Question: Show name, country, age for all singers ordered by age from the oldest to the youngest.

Gold SQL:
 SELECT name ,  country ,  age FROM singer ORDER BY age DESC

Base model SQL:
 SELECT singer.Name, singer.Country, singer.Age
FROM singer
ORDER BY singer.Age DESC;

Tuned model SQL:
 SELECT name ,  country ,  age FROM singer ORDER BY age
Example: 3
DB ID: concert_singer
Question: What are the names, countries, and ages for every singer in descending or

## 17A. Qualitative Check on all Dev Examples

We inspect all dev questions and compare the gold SQL, the original base-model output, and the fine-tuned LoRA output.

In [ ]:
import pandas as pd
import sqlite3
import re

def normalize_sql_for_match(sql_text: str) -> str:
    sql = str(sql_text).strip().lower()
    sql = sql.replace(";", "")
    sql = re.sub(r"\s+", " ", sql)
    sql = re.sub(r"\s*,\s*", ", ", sql)
    sql = re.sub(r"\s*\(\s*", "(", sql)
    sql = re.sub(r"\s*\)\s*", ")", sql)
    sql = re.sub(r"\s*=\s*", " = ", sql)
    sql = re.sub(r"\s+", " ", sql).strip()
    return sql

def execute_sql(db_path, sql):
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        return True, rows, None
    except Exception as e:
        return False, None, str(e)

full_rows = []

validation_examples = processed_dev[:run_config["dev_limit"]]

for i, ex in enumerate(validation_examples):
    prompt = ex["prompt"]

    base_sql = generate_sql_with_model(
        prompt, base_model_for_compare, base_tokenizer_for_compare, max_new_tokens=128,
    )
    tuned_sql = generate_sql_with_model(
        prompt, eval_model, eval_tokenizer, max_new_tokens=128,
    )

    gold_sql = ex["target_sql"]

    gold_norm = normalize_sql_for_match(gold_sql)
    base_norm = normalize_sql_for_match(base_sql)
    tuned_norm = normalize_sql_for_match(tuned_sql)

    base_exact = base_norm == gold_norm
    tuned_exact = tuned_norm == gold_norm

    gold_ok, gold_res, gold_err = execute_sql(ex["sqlite_path"], gold_sql)
    base_ok, base_res, base_err = execute_sql(ex["sqlite_path"], base_sql)
    tuned_ok, tuned_res, tuned_err = execute_sql(ex["sqlite_path"], tuned_sql)

    base_exec = gold_ok and base_ok and (gold_res == base_res)
    tuned_exec = gold_ok and tuned_ok and (gold_res == tuned_res)

    # ── Execution-guided fallback (per example, inside loop) ──────────────────
    # Use tuned SQL if it executes at all; fall back to base if tuned fails;
    # if both fail, keep tuned (it may still be more correct structurally)
    if tuned_ok:
        final_sql = tuned_sql
    elif base_ok:
        final_sql = base_sql
    else:
        final_sql = tuned_sql

    final_ok, final_res, final_err = execute_sql(ex["sqlite_path"], final_sql)
    final_exec = gold_ok and final_ok and (gold_res == final_res)
    # ──────────────────────────────────────────────────────────────────────────

    if (not base_exec) and tuned_exec:
        outcome = "tuned_better_exec"
    elif base_exec and (not tuned_exec):
        outcome = "tuned_worse_exec"
    elif base_exec and tuned_exec:
        outcome = "same_correct_exec"
    elif (not base_exec) and (not tuned_exec) and ((not base_exact) and tuned_exact):
        outcome = "tuned_better_exact_only"
    elif (not base_exec) and (not tuned_exec) and (base_exact and (not tuned_exact)):
        outcome = "tuned_worse_exact_only"
    else:
        outcome = "same_wrong_exec"

    full_rows.append({
        "example_id": i,
        "db_id": ex["db_id"],
        "question": ex["question"],
        "gold_sql": gold_sql,
        "base_sql": base_sql,
        "tuned_sql": tuned_sql,
        "final_sql": final_sql,           # fallback result
        "sqlite_path": ex["sqlite_path"],
        "gold_norm": gold_norm,
        "base_norm": base_norm,
        "tuned_norm": tuned_norm,
        "base_exact_match": base_exact,
        "tuned_exact_match": tuned_exact,
        "gold_exec_ok": gold_ok,
        "base_exec_ok": base_ok,
        "tuned_exec_ok": tuned_ok,
        "base_exec_match": base_exec,
        "tuned_exec_match": tuned_exec,
        "final_exec_match": final_exec,   # fallback execution result
        "base_error": base_err,
        "tuned_error": tuned_err,
        "outcome": outcome,
    })

    if (i + 1) % 20 == 0:
        print(f"Finished {i + 1}/{len(validation_examples)} examples")

full_eval_df = pd.DataFrame(full_rows)

print("Total examples:", len(full_eval_df))
print("Base exact matches:        ", int(full_eval_df["base_exact_match"].sum()))
print("Tuned exact matches:       ", int(full_eval_df["tuned_exact_match"].sum()))
print("Base execution matches:    ", int(full_eval_df["base_exec_match"].sum()))
print("Tuned execution matches:   ", int(full_eval_df["tuned_exec_match"].sum()))
print("Fallback execution matches:", int(full_eval_df["final_exec_match"].sum()))
print()
print("Outcome counts:")
print(full_eval_df["outcome"].value_counts())

full_eval_csv = OUTPUTDIR / "milestone1_full_eval.csv"
full_eval_df.to_csv(full_eval_csv, index=False)
print("Saved:", full_eval_csv)

KeyboardInterrupt: 

## 17B. Summary metrics for all dev examples

This is the key result: if the tuned model helps

In [ ]:
total = len(full_eval_df)
base_matches = int(full_eval_df["base_exact_match"].sum())
tuned_matches = int(full_eval_df["tuned_exact_match"].sum())

print("Total examples:", total)
print("Base exact matches:", base_matches)
print("Tuned exact matches:", tuned_matches)
print("Base accuracy:", round(base_matches / total, 4))
print("Tuned accuracy:", round(tuned_matches / total, 4))

print("\nOutcome counts:")
print(full_eval_df["outcome"].value_counts())

NameError: name 'full_eval_df' is not defined

## 17C. Inspect key example differences

To see only the examples where the tuned model changed something

In [ ]:
diff_df = full_eval_df[full_eval_df["base_norm"] != full_eval_df["tuned_norm"]].copy()

print("Number of changed outputs:", len(diff_df))

diff_df[[
    "example_id",
    "db_id",
    "question",
    "gold_sql",
    "base_sql",
    "tuned_sql",
    "outcome",
]].head(20)

# Correct outcome label names matching Cell 47 logic
better_df = full_eval_df[
    full_eval_df["outcome"].isin(["tuned_better_exec", "tuned_better_exact_only"])
].copy()

worse_df = full_eval_df[
    full_eval_df["outcome"].isin(["tuned_worse_exec", "tuned_worse_exact_only"])
].copy()

print("\nTuned better (exec or exact):", len(better_df))
print("Tuned worse  (exec or exact):", len(worse_df))
print("Same correct:", len(full_eval_df[full_eval_df["outcome"] == "same_correct_exec"]))
print("Same wrong:  ", len(full_eval_df[full_eval_df["outcome"] == "same_wrong_exec"]))

NameError: name 'full_eval_df' is not defined

## 17D. Saving our observations

Saving our observations as a CSV

In [ ]:
full_eval_csv = OUTPUTDIR / "milestone1_full_dev_evaluation.csv"
full_eval_df.to_csv(full_eval_csv, index=False)
print("Saved full evaluation to:", full_eval_csv)

Saved full evaluation to: /content/drive/MyDrive/CS_512_Project/milestone1_lora/csv_outputs/milestone1_full_dev_evaluation.csv


## 18. Save Milestone 1 Validation Results

We store the qualitative comparison in a CSV so it can be referenced in the report and reused in later milestones.

In [ ]:
milestone1_df = pd.DataFrame(milestone1_rows)

milestone1_csv = OUTPUTDIR / "milestone1_qualitative_validation.csv"
milestone1_df.to_csv(milestone1_csv, index=False)

print(milestone1_df[["example_id", "db_id", "question"]])
print("\nSaved qualitative validation file to:", milestone1_csv)

   example_id           db_id  \
0           0  concert_singer   
1           1  concert_singer   
2           2  concert_singer   
3           3  concert_singer   
4           4  concert_singer   

                                            question  
0                       How many singers do we have?  
1               What is the total number of singers?  
2  Show name, country, age for all singers ordere...  
3  What are the names, countries, and ages for ev...  
4  What is the average, minimum, and maximum age ...  

Saved qualitative validation file to: /content/drive/MyDrive/CS_512_Project/milestone1_lora/csv_outputs/milestone1_qualitative_validation.csv


More Checks

In [ ]:
import sqlite3
import pandas as pd

def exec_sql(db_path, sql):
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute(sql)
        res = cur.fetchall()
        conn.close()
        return True, res
    except Exception as e:
        return False, str(e)

def normalize_sql_for_match(sql_text: str) -> str:
    return " ".join(sql_text.strip().lower().replace(";", "").split())

In [ ]:
rows = []
for _, ex in milestone1_df.iterrows():
    gold_ok, gold_res = exec_sql(ex["sqlite_path"], ex["gold_sql"])
    base_ok, base_res = exec_sql(ex["sqlite_path"], ex["base_sql"])
    tuned_ok, tuned_res = exec_sql(ex["sqlite_path"], ex["tuned_sql"])

    rows.append({
        "example_id": ex["example_id"],
        "db_id": ex["db_id"],
        "question": ex["question"],
        "gold_sql": ex["gold_sql"],
        "base_sql": ex["base_sql"],
        "tuned_sql": ex["tuned_sql"],
        "gold_exec_ok": gold_ok,
        "base_exec_ok": base_ok,
        "tuned_exec_ok": tuned_ok,
        "base_exec_match": gold_ok and base_ok and base_res == gold_res,
        "tuned_exec_match": gold_ok and tuned_ok and tuned_res == gold_res,
    })

exec_df = pd.DataFrame(rows)

print("Base execution matches:", int(exec_df["base_exec_match"].sum()), "/", len(exec_df))
print("Tuned execution matches:", int(exec_df["tuned_exec_match"].sum()), "/", len(exec_df))

Base execution matches: 5 / 5
Tuned execution matches: 4 / 5


## 19. Optional Milestone 1 Summary Metrics

For a lightweight milestone-level check, we compute exact string match counts against the gold SQL on the small validation sample.

In [ ]:
def normalize_sql_for_match(sql_text: str) -> str:
    return " ".join(sql_text.strip().lower().split())

milestone1_df["gold_norm"] = milestone1_df["gold_sql"].apply(normalize_sql_for_match)
milestone1_df["base_norm"] = milestone1_df["base_sql"].apply(normalize_sql_for_match)
milestone1_df["tuned_norm"] = milestone1_df["tuned_sql"].apply(normalize_sql_for_match)

milestone1_df["base_exact_match"] = milestone1_df["base_norm"] == milestone1_df["gold_norm"]
milestone1_df["tuned_exact_match"] = milestone1_df["tuned_norm"] == milestone1_df["gold_norm"]

print("Base exact matches:", int(milestone1_df["base_exact_match"].sum()), "/", len(milestone1_df))
print("Tuned exact matches:", int(milestone1_df["tuned_exact_match"].sum()), "/", len(milestone1_df))

milestone1_metrics_csv = OUTPUTDIR / "milestone1_metrics_preview.csv"
milestone1_df.to_csv(milestone1_metrics_csv, index=False)

print("Saved metrics preview to:", milestone1_metrics_csv)

Base exact matches: 0 / 5
Tuned exact matches: 4 / 5
Saved metrics preview to: /content/drive/MyDrive/CS_512_Project/milestone1_lora/csv_outputs/milestone1_metrics_preview.csv


In [ ]:
# import os
# # Search Drive for any .sqlite file to confirm the real path
# for root, dirs, files in os.walk("/content/drive/MyDrive/CS_512_Project"):
#     for f in files:
#         if f.endswith(".sqlite"):
#             print(os.path.join(root, f))
#             break
#     else:
#         continue
#     break


# #print(base_model_for_compare is eval_model.base_model.model)  # will print True